# Day 58 — Code review, refactoring, and maintainability
Objectives:
- Refactor notebook code into reusable modules.
- Add unit tests and simple CI checklist.
- Improve documentation (docstrings, README sections).
- Prepare a maintainers checklist for DS repos.

## 1) From notebook to module
Take code from prior days (e.g., your preprocessing + training) and move it into a `src/` package.
Example layout:
````
ds-60day/
  src/
    __init__.py
    data.py        # load/clean functions
    features.py    # feature engineering
    model.py       # train/evaluate/save
  tests/
    test_model.py
  notebooks/
    ...
````


In [ ]:
# Example refactor target
from dataclasses import dataclass
from typing import Tuple
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

@dataclass
class TrainResult:
    pipeline: Pipeline
    auc: float

def build_pipeline() -> Pipeline:
    pre = ColumnTransformer([('cat', OneHotEncoder(handle_unknown='ignore'), ['sex','class']),
                             ('num', StandardScaler(), ['fare','age'])])
    return Pipeline([('pre', pre), ('clf', LogisticRegression(max_iter=1000))])

def train_evaluate(df: pd.DataFrame) -> TrainResult:
    X = df[['sex','class','fare','age']]
    y = df['survived']
    Xtr,Xte,ytr,yte = train_test_split(X,y, test_size=0.2, random_state=42, stratify=y)
    pipe = build_pipeline()
    pipe.fit(Xtr,ytr)
    auc = roc_auc_score(yte, pipe.predict_proba(Xte)[:,1])
    return TrainResult(pipe, auc)


## 2) Tests with pytest
Create `tests/test_model.py` with unit tests for your functions.
Example:
```python
import pytest, pandas as pd, seaborn as sns
from src.model import train_evaluate
def test_train_evaluate_runs():
    df = sns.load_dataset('titanic').dropna(subset=['survived','sex','class','fare','age'])
    res = train_evaluate(df)
    assert 0.5 <= res.auc <= 1.0
```
Run: `pytest -q`

## 3) Document with docstrings & README
- Ensure every function has a concise docstring (inputs/outputs, assumptions).
- Add a short project README describing how to run training and tests.

## 4) Maintainability checklist
- Style & linting: black, flake8, mypy
- Tests pass: pytest
- Reproducibility: pinned requirements, deterministic seeds
- Data contracts: pandera schemas on inputs/outputs
- Logging and error handling
- CI: run pytest + lint on PR

## Exercises
1) Move at least two functions into `src/` and add tests.
2) Add type hints and docstrings; run mypy.
3) Write a short maintainers guide (markdown) in your repo root.